# Telecom Customer Churn Prediction & Behaviour Analytics

**End-to-end project using the three supplied files:** `usage_data.csv`, `customer_info.csv`, and `churn_labels.csv`.

## Project objectives
1. Inspect each raw file independently and visualize its distributions and data quality.
2. Clean duplicate customer records and aggregate monthly usage to **one row per customer** before joining. This avoids treating 12 monthly records from one person as 12 independent customers.
3. Measure customer-level class imbalance and apply SMOTE **only to training data**.
4. Preprocess missing values and categorical variables, compare three algorithms, and select a model based on validation metrics.
5. Report accuracy, precision, recall, F1, ROC-AUC, PR-AUC, confusion matrix and classification report.
6. Explain influential features and analyze churn behaviour by contract, region, demographics, usage and complaints.

> **Important interpretation note:** this dataset contains 12 monthly usage rows per customer in 2023, plus a customer-level churn label. We aggregate the usage history to customer-level features. For a real prospective deployment, features must be restricted to information available before the churn prediction date; if the labels refer to churn during 2023, using all 12 months could introduce temporal leakage. This notebook is a retrospective class project unless a prediction cutoff date is defined.

## How to run
Run cells from top to bottom. If the three CSVs are not already in the Colab `/content` directory, the first data-loading cell will let you upload them.

## 1. Install / import libraries

In [ ]:
# If running in a fresh Colab runtime, uncomment this line if needed:
# %pip install -q imbalanced-learn

import os, warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
plt.rcParams.update({"figure.figsize": (8, 5), "figure.dpi": 120,
                     "axes.titlesize": 13, "axes.labelsize": 11,
                     "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})
print("Libraries ready. pandas:", pd.__version__)

## 2. Load the three supplied files

In [ ]:
# Expected file names in the Colab working directory.
# If they are missing, upload the three CSV files when prompted.
expected = ["usage_data.csv", "customer_info.csv", "churn_labels.csv"]
if not all(os.path.exists(f"/content/{f}") for f in expected):
    try:
        from google.colab import files
        print("Upload usage_data.csv, customer_info.csv, and churn_labels.csv")
        uploaded = files.upload()
    except Exception:
        print("Place the three CSV files in the current working directory and rerun this cell.")

# Resolve exact names, including the (3) suffix sometimes added by browsers.
def find_csv(starts_with):
    candidates = [f for f in os.listdir("/content") if f.lower().endswith(".csv") and f.lower().startswith(starts_with)]
    if not candidates:
        candidates = [f for f in os.listdir(".") if f.lower().endswith(".csv") and f.lower().startswith(starts_with)]
    if not candidates:
        raise FileNotFoundError(f"Could not find a CSV starting with {starts_with!r}. Upload/rename the file.")
    return os.path.join("/content", candidates[0]) if os.path.exists(os.path.join("/content", candidates[0])) else candidates[0]

usage_path = find_csv("usage_data")
customer_path = find_csv("customer_info")
labels_path = find_csv("churn_labels")
usage_data = pd.read_csv(usage_path)
customer_info = pd.read_csv(customer_path)
churn_labels = pd.read_csv(labels_path)

print("Usage data:", usage_data.shape, "|", usage_path)
print("Customer info:", customer_info.shape, "|", customer_path)
print("Churn labels:", churn_labels.shape, "|", labels_path)
display(usage_data.head(3), customer_info.head(3), churn_labels.head(3))

## 3. Inspect the raw files separately

In [ ]:
for name, data in [("usage_data", usage_data), ("customer_info", customer_info), ("churn_labels", churn_labels)]:
    print(f"\n--- {name} ---")
    print("Shape:", data.shape)
    print("Duplicate full rows:", data.duplicated().sum())
    print("Missing cells:", int(data.isna().sum().sum()))
    print("Unique CustomerIDs:", data["CustomerID"].nunique() if "CustomerID" in data else "N/A")
    display(data.isna().sum().rename("missing_count").to_frame())
    display(data.head(2))

In [ ]:
# Usage variables: distributions
usage_numeric = [c for c in ["CallMinutes", "DataUsageGB", "SMSCount", "Complaints"] if c in usage_data]
usage_data[usage_numeric].hist(bins=30, figsize=(12, 8), edgecolor="white")
plt.suptitle("Raw monthly usage: feature distributions", y=1.02)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, ax in zip(usage_numeric, axes.ravel()):
    ax.boxplot(usage_data[col].dropna(), vert=False)
    ax.set_title(f"{col}: outlier spread")
    ax.set_xlabel(col)
plt.tight_layout(); plt.show()

In [ ]:
# Customer profile categorical distributions and numeric distributions
cat_profile = [c for c in ["Gender", "Region", "ContractType"] if c in customer_info]
fig, axes = plt.subplots(1, max(1, len(cat_profile)), figsize=(5*max(1, len(cat_profile)), 4))
if len(cat_profile) == 1: axes = [axes]
for col, ax in zip(cat_profile, axes):
    customer_info[col].fillna("Missing").value_counts().plot(kind="bar", ax=ax)
    ax.set_title(f"Customer profile: {col}"); ax.set_ylabel("Rows"); ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

numeric_profile = [c for c in ["Age", "MonthlyCharges"] if c in customer_info]
if numeric_profile:
    customer_info[numeric_profile].hist(bins=25, figsize=(10, 4), edgecolor="white")
    plt.suptitle("Customer profile: numeric distributions", y=1.02)
    plt.tight_layout(); plt.show()

# Churn label distribution in the label file
label_counts = churn_labels["Churn"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot(kind="bar", ax=ax, color=["#3977a8", "#d97757"][:len(label_counts)])
ax.set_title("Raw churn label counts (customer level)"); ax.set_xlabel("Churn (0=Stayed, 1=Churned)"); ax.set_ylabel("Customers")
for i, v in enumerate(label_counts.values): ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout(); plt.show()
print("Customer-level churn percentages:")
print((churn_labels["Churn"].value_counts(normalize=True).sort_index()*100).round(2))

## 4. Clean customer information and aggregate usage to customer level

The usage file has repeated monthly observations (12 per customer). We must summarize these before joining to customer-level churn labels. The customer-info file has exact duplicate rows and a small number of conflicting duplicate IDs. We remove exact duplicates, then keep the most complete record for any remaining duplicate ID; ambiguous conflicting records are reported for transparency.

In [ ]:
# Standardize basic keys and remove exact duplicate rows.
for _df in (usage_data, customer_info, churn_labels):
    _df["CustomerID"] = _df["CustomerID"].astype(str).str.strip()

usage_data = usage_data.drop_duplicates().copy()
customer_info = customer_info.drop_duplicates().copy()
churn_labels = churn_labels.drop_duplicates().copy()

# Parse dates where available
if "Month" in usage_data:
    usage_data["Month"] = pd.to_datetime(usage_data["Month"], errors="coerce")
if "SignupDate" in customer_info:
    customer_info["SignupDate"] = pd.to_datetime(customer_info["SignupDate"], errors="coerce")

# Report customer IDs that have conflicting customer profiles before resolving them.
conflict_ids = customer_info.loc[customer_info["CustomerID"].duplicated(keep=False), "CustomerID"].unique()
print("Customer-info rows after exact-duplicate removal:", len(customer_info))
print("IDs still repeated:", len(conflict_ids))
if len(conflict_ids):
    display(customer_info[customer_info["CustomerID"].isin(conflict_ids)].sort_values("CustomerID"))

# Keep the most complete row per ID; for equal completeness, retain the first source row.
customer_info["_missing_count"] = customer_info.isna().sum(axis=1)
customer_info_clean = (customer_info.sort_values(["CustomerID", "_missing_count"])
                       .drop_duplicates("CustomerID", keep="first")
                       .drop(columns="_missing_count").copy())
print("Customer info cleaned:", customer_info_clean.shape,
      "| duplicate IDs remaining:", customer_info_clean["CustomerID"].duplicated().sum())

# Create customer-level usage features from the monthly usage history.
usage_features = [c for c in ["CallMinutes", "DataUsageGB", "SMSCount", "Complaints"] if c in usage_data]
agg_dict = {}
for col in usage_features:
    agg_dict[col] = ["mean", "std", "min", "max", "sum"]
usage_agg = usage_data.groupby("CustomerID").agg(agg_dict)
usage_agg.columns = [f"{col}_{stat}" for col, stat in usage_agg.columns]
usage_agg = usage_agg.reset_index()

# Add simple temporal features: first/last month values and change over the observed year.
if "Month" in usage_data and usage_features:
    sorted_usage = usage_data.sort_values(["CustomerID", "Month"])
    for col in [c for c in ["CallMinutes", "DataUsageGB", "SMSCount", "Complaints"] if c in usage_data]:
        first = sorted_usage.groupby("CustomerID")[col].first().rename(f"{col}_first_month")
        last = sorted_usage.groupby("CustomerID")[col].last().rename(f"{col}_last_month")
        usage_agg = usage_agg.merge(first, on="CustomerID", how="left")
        usage_agg = usage_agg.merge(last, on="CustomerID", how="left")
        usage_agg[f"{col}_change_first_to_last"] = usage_agg[f"{col}_last_month"] - usage_agg[f"{col}_first_month"]

# Count observed months for data-quality / coverage context.
if "Month" in usage_data:
    month_counts = usage_data.groupby("CustomerID")["Month"].nunique().rename("months_observed").reset_index()
    usage_agg = usage_agg.merge(month_counts, on="CustomerID", how="left")

print("Usage monthly rows:", len(usage_data), "| unique customers:", usage_data.CustomerID.nunique())
print("Aggregated usage features:", usage_agg.shape)
display(usage_agg.head())

In [ ]:
# Check labels have one valid target per customer.
print("Label rows:", len(churn_labels), "| unique IDs:", churn_labels.CustomerID.nunique())
print("Duplicate IDs in labels:", churn_labels.CustomerID.duplicated().sum())
print("Churn values:", sorted(churn_labels["Churn"].dropna().unique().tolist()))
assert churn_labels["CustomerID"].is_unique, "Churn labels must have one row per customer."
assert set(churn_labels["Churn"].dropna().unique()).issubset({0, 1}), "Churn must be encoded as 0/1."

# Merge only after usage has been aggregated to one row per customer.
df = customer_info_clean.merge(usage_agg, on="CustomerID", how="inner", validate="one_to_one")
df = df.merge(churn_labels[["CustomerID", "Churn"]], on="CustomerID", how="inner", validate="one_to_one")
print("Final customer-level modeling table:", df.shape)
print("Unique customers:", df.CustomerID.nunique())
print("Missing churn labels:", df.Churn.isna().sum())
print("Customer IDs present in all three sources:", len(df))
display(df.head())

## 5. Data-quality checks after merging

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean()*100).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values("missing_percent", ascending=False)
display(quality)
print("Duplicate full rows in final table:", df.duplicated().sum())
print("Customers by number of observed usage months:")
if "months_observed" in df:
    display(df["months_observed"].value_counts().sort_index().to_frame("customers"))

fig, ax = plt.subplots(figsize=(9, 5))
miss = quality[quality.missing_count > 0].sort_values("missing_percent", ascending=True)
if len(miss):
    ax.barh(miss.index, miss.missing_percent)
    ax.set_xlabel("Missing values (%)"); ax.set_title("Missingness in the merged customer-level table")
else:
    ax.text(.5, .5, "No missing values", ha="center", va="center")
plt.tight_layout(); plt.show()

## 6. Class imbalance at the correct level (one row per customer)

In [ ]:
class_counts = df["Churn"].value_counts().sort_index()
class_pct = df["Churn"].value_counts(normalize=True).sort_index()*100
class_summary = pd.DataFrame({"customers": class_counts, "percent": class_pct.round(2)})
display(class_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(["Stayed (0)", "Churned (1)"], [class_counts.get(0,0), class_counts.get(1,0)], color=["#3977a8", "#d97757"])
axes[0].set_title("Customer-level class counts"); axes[0].set_ylabel("Customers")
for i, v in enumerate([class_counts.get(0,0), class_counts.get(1,0)]): axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")
axes[1].pie([class_counts.get(0,0), class_counts.get(1,0)], labels=["Stayed", "Churned"], autopct="%.1f%%", startangle=90, colors=["#3977a8", "#d97757"])
axes[1].set_title("Class proportions")
plt.tight_layout(); plt.show()
print(f"Class ratio (majority/minority): {class_counts.max()/class_counts.min():.2f}:1")
print("Note: the class distribution is computed after aggregation, so each customer contributes once.")

## 7. Behaviour analysis before modeling

In [ ]:
# Churn rate by customer profile categories.
for col in ["ContractType", "Region", "Gender"]:
    if col in df.columns:
        rates = df.groupby(col, dropna=False)["Churn"].agg(["mean", "count"]).sort_values("mean", ascending=False)
        rates["churn_rate_pct"] = rates["mean"]*100
        print(f"\nChurn rate by {col}:")
        display(rates[["count", "churn_rate_pct"]].round(2))
        fig, ax = plt.subplots(figsize=(7, 4))
        rates["churn_rate_pct"].sort_values().plot(kind="barh", ax=ax, color="#d97757")
        ax.set_title(f"Churn rate by {col}"); ax.set_xlabel("Churn rate (%)"); ax.set_ylabel(col)
        plt.tight_layout(); plt.show()

# Numeric distributions split by churn, focusing on core usage and price variables.
for col in ["Age", "MonthlyCharges", "CallMinutes_mean", "DataUsageGB_mean", "SMSCount_mean", "Complaints_mean", "Complaints_sum"]:
    if col in df.columns:
        fig, ax = plt.subplots(figsize=(7, 4))
        for label, color in [(0, "#3977a8"), (1, "#d97757")]:
            vals = df.loc[df.Churn == label, col].dropna()
            if len(vals): ax.hist(vals, bins=30, alpha=.55, density=True, label=f"Churn={label}", color=color)
        ax.set_title(f"{col}: distribution by churn label"); ax.set_xlabel(col); ax.set_ylabel("Density"); ax.legend()
        plt.tight_layout(); plt.show()

# Correlation heatmap for numeric features plus churn (matplotlib only).
num_df = df.select_dtypes(include=np.number).copy()
if num_df.shape[1] > 1:
    corr = num_df.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(12, 9))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(corr.columns))); ax.set_yticklabels(corr.columns, fontsize=7)
    ax.set_title("Numeric feature correlation matrix")
    fig.colorbar(im, ax=ax, shrink=.75, label="Correlation")
    plt.tight_layout(); plt.show()

## 8. Prepare features and train/test split

`CustomerID` is an identifier, not a predictive feature, so it is excluded. We also exclude raw date strings after extracting signup tenure. Categorical missing values are imputed and one-hot encoded; numeric missing values are median-imputed and scaled. The test set remains untouched by SMOTE.

In [ ]:
# Derive customer tenure at the end of the usage observation window.
model_df = df.copy()
if "SignupDate" in model_df.columns:
    reference_date = usage_data["Month"].max() if "Month" in usage_data and usage_data["Month"].notna().any() else pd.Timestamp("2023-12-31")
    model_df["CustomerTenureDays"] = (reference_date - model_df["SignupDate"]).dt.days
    model_df.loc[model_df["CustomerTenureDays"] < 0, "CustomerTenureDays"] = np.nan

# Remove identifiers / raw dates, target, and any non-predictive raw Month fields.
drop_cols = [c for c in ["CustomerID", "Churn", "SignupDate"] if c in model_df.columns]
X = model_df.drop(columns=drop_cols)
y = model_df["Churn"].astype(int)

# Remove columns that are entirely missing or constant.
constant_cols = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
if constant_cols:
    print("Dropping constant/all-empty columns:", constant_cols)
    X = X.drop(columns=constant_cols)

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
print("Modeling rows:", len(X), "| features:", X.shape[1])
print("Numeric features:", len(numeric_features), numeric_features)
print("Categorical features:", len(categorical_features), categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class counts before SMOTE:\n", y_train.value_counts().sort_index().to_dict())
print("Test class counts (untouched):\n", y_test.value_counts().sort_index().to_dict())

numeric_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
], remainder="drop", verbose_feature_names_out=False)
print("Preprocessing pipeline configured.")

## 9. Apply SMOTE to training data only

In [ ]:
# Fit preprocessing only on training data, then resample the transformed training rows.
# The test data is not resampled and is never used to fit the preprocessor or SMOTE.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Before SMOTE:", pd.Series(y_train).value_counts().sort_index().to_dict())
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)
print("After SMOTE:", pd.Series(y_train_smote).value_counts().sort_index().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pre_counts = pd.Series(y_train).value_counts().sort_index()
post_counts = pd.Series(y_train_smote).value_counts().sort_index()
axes[0].bar(["Stayed", "Churned"], [pre_counts.get(0,0), pre_counts.get(1,0)], color=["#3977a8", "#d97757"])
axes[0].set_title("Training classes before SMOTE")
axes[1].bar(["Stayed", "Churned"], [post_counts.get(0,0), post_counts.get(1,0)], color=["#3977a8", "#d97757"])
axes[1].set_title("Training classes after SMOTE")
for ax in axes: ax.set_ylabel("Rows")
plt.tight_layout(); plt.show()
print("SMOTE balances only the training set; the test distribution remains naturally imbalanced.")

## 10. Train and compare three algorithms

We compare Logistic Regression (linear baseline), Random Forest (bagged trees), and Extra Trees (highly randomized trees). All three use the same train/test split and the same SMOTE-resampled training set. The preprocessor and resampling happen before model fitting; no synthetic rows enter the test set.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight=None, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE, class_weight=None),
    "Extra Trees": ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE, class_weight=None)
}

fitted_models = {}
rows = []
probabilities = {}
predictions = {}
for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    pred = model.predict(X_test_processed)
    proba = model.predict_proba(X_test_processed)[:, 1]
    fitted_models[name] = model
    predictions[name] = pred
    probabilities[name] = proba
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, proba),
        "PR_AUC": average_precision_score(y_test, proba)
    })

results = pd.DataFrame(rows).sort_values(["F1", "Recall", "PR_AUC"], ascending=False).reset_index(drop=True)
display(results.style.format({c: "{:.3f}" for c in ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]}))
best_name = results.iloc[0]["Model"]
best_model = fitted_models[best_name]
print("Selected model by highest F1, then recall and PR-AUC:", best_name)
print("F1 balances precision and recall; PR-AUC remains important because churn is uncommon. Inspect all metrics before accepting a model.")

## 11. Evaluation plots: crisp confusion matrices, ROC and precision-recall curves

In [ ]:
# Confusion matrices for all three models with count annotations.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, models.keys()):
    cm = confusion_matrix(y_test, predictions[name], labels=[0, 1])
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
    ax.set_xticks([0,1]); ax.set_xticklabels(["Stayed (0)", "Churned (1)"])
    ax.set_yticks([0,1]); ax.set_yticklabels(["Stayed (0)", "Churned (1)"])
    threshold = cm.max()/2 if cm.max() else 0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                    color="white" if cm[i,j] > threshold else "black", fontsize=11, fontweight="bold")
fig.suptitle("Confusion matrices — test set (not resampled)", y=1.03)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for name in models:
    RocCurveDisplay.from_predictions(y_test, probabilities[name], name=name, ax=ax)
ax.plot([0,1],[0,1], linestyle="--", color="gray", linewidth=1)
ax.set_title("ROC curves on untouched test set")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for name in models:
    PrecisionRecallDisplay.from_predictions(y_test, probabilities[name], name=name, ax=ax)
ax.set_title("Precision–Recall curves on untouched test set")
plt.tight_layout(); plt.show()

print("Best model classification report:")
print(classification_report(y_test, predictions[best_name], target_names=["Stayed", "Churned"], digits=3, zero_division=0))
print("Best model confusion matrix (rows=true, columns=predicted; order [0,1]):")
print(confusion_matrix(y_test, predictions[best_name], labels=[0,1]))

## 12. Feature importance / which features matter?

In [ ]:
# Get transformed feature names and model importance where supported.
feature_names = preprocessor.get_feature_names_out()
importance = None
if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False)
elif hasattr(best_model, "coef_"):
    importance = pd.Series(np.abs(best_model.coef_[0]), index=feature_names).sort_values(ascending=False)

if importance is not None:
    top_importance = importance.head(20).sort_values()
    fig, ax = plt.subplots(figsize=(9, 7))
    top_importance.plot(kind="barh", ax=ax, color="#3977a8")
    ax.set_title(f"Top 20 transformed features — {best_name}")
    ax.set_xlabel("Importance" if hasattr(best_model, "feature_importances_") else "Absolute coefficient magnitude")
    plt.tight_layout(); plt.show()
    display(importance.head(20).rename("importance").to_frame())
else:
    print("This model does not expose built-in feature importance.")

# Permutation importance on the untouched test set is model-agnostic.
from sklearn.inspection import permutation_importance
perm = permutation_importance(best_model, X_test_processed, y_test, scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=feature_names).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 7))
perm_imp.head(20).sort_values().plot(kind="barh", ax=ax, color="#d97757")
ax.set_title("Permutation importance on test set (PR-AUC decrease)")
ax.set_xlabel("Mean score decrease after shuffling feature")
plt.tight_layout(); plt.show()
display(perm_imp.head(20).rename("mean_PR_AUC_drop").to_frame())
print("Interpretation: importance indicates predictive usefulness in this dataset, not causation.")

## 13. Threshold tuning for churn detection

A 0.50 probability threshold is not always ideal when missing churners is costly. We select a threshold using the validation/test predictions for illustration; for a production-grade unbiased estimate, threshold tuning should use a separate validation split or cross-validation, and the final test set should remain untouched. This section is a project demonstration and should be described accordingly.

In [ ]:
from sklearn.metrics import precision_recall_curve
best_proba = probabilities[best_name]
precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, best_proba)
# F1 for thresholds returned by precision_recall_curve (last precision/recall point has no threshold)
f1_curve = (2 * precision_curve[:-1] * recall_curve[:-1] /
            (precision_curve[:-1] + recall_curve[:-1] + 1e-12))
if len(thresholds):
    best_idx = int(np.nanargmax(f1_curve))
    chosen_threshold = float(thresholds[best_idx])
    print(f"Illustrative threshold maximizing test-set F1: {chosen_threshold:.3f}")
    tuned_pred = (best_proba >= chosen_threshold).astype(int)
    print(classification_report(y_test, tuned_pred, target_names=["Stayed", "Churned"], digits=3, zero_division=0))
    print("Tuned-threshold confusion matrix:")
    print(confusion_matrix(y_test, tuned_pred, labels=[0,1]))
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(thresholds, precision_curve[:-1], label="Precision")
    ax.plot(thresholds, recall_curve[:-1], label="Recall")
    ax.plot(thresholds, f1_curve, label="F1")
    ax.axvline(chosen_threshold, color="gray", linestyle="--", label=f"Chosen={chosen_threshold:.2f}")
    ax.set_xlabel("Probability threshold"); ax.set_ylabel("Score"); ax.set_ylim(0,1.02)
    ax.set_title("Threshold trade-off (illustrative)"); ax.legend()
    plt.tight_layout(); plt.show()

## 14. Customer behaviour insights and practical recommendations

In [ ]:
# Compact summary tables for business interpretation.
print("Overall customer-level churn rate:", round(df["Churn"].mean()*100, 2), "%")

for col in ["ContractType", "Region", "Gender"]:
    if col in df:
        summary = df.groupby(col, dropna=False).agg(
            customers=("Churn", "size"),
            churners=("Churn", "sum"),
            churn_rate=("Churn", "mean")
        )
        summary["churn_rate_pct"] = (summary["churn_rate"]*100).round(2)
        print(f"\n{col}:")
        display(summary[["customers", "churners", "churn_rate_pct"]].sort_values("churn_rate_pct", ascending=False))

# Usage and complaint behaviour table by churn class.
behaviour_features = [c for c in ["CallMinutes_mean", "DataUsageGB_mean", "SMSCount_mean", "Complaints_mean", "Complaints_sum", "MonthlyCharges", "CustomerTenureDays"] if c in df]
if behaviour_features:
    behaviour_summary = df.groupby("Churn")[behaviour_features].agg(["mean", "median"])
    display(behaviour_summary.round(2))

# Plot churn rate by complaint count per month (use mean complaints as customer-level feature).
if "Complaints_mean" in df:
    temp = df[["Complaints_mean", "Churn"]].copy()
    temp["ComplaintBand"] = pd.cut(temp["Complaints_mean"], bins=[-0.01, 0, 0.5, 1.5, 3, np.inf], labels=["0", "0–0.5", "0.5–1.5", "1.5–3", "3+"])
    band_rates = temp.groupby("ComplaintBand", observed=False)["Churn"].agg(["mean", "count"])
    band_rates["churn_rate_pct"] = band_rates["mean"]*100
    display(band_rates.round(3))
    fig, ax = plt.subplots(figsize=(7,4))
    band_rates["churn_rate_pct"].plot(kind="bar", ax=ax, color="#d97757")
    ax.set_title("Churn rate by average monthly complaints")
    ax.set_xlabel("Average complaint band"); ax.set_ylabel("Churn rate (%)"); ax.tick_params(axis="x", rotation=20)
    plt.tight_layout(); plt.show()

print("\nRecommendations should be tied to actual group differences above:")
print("1. Prioritize high-risk customers identified by the chosen model for retention outreach.")
print("2. If churn is higher for short-term contracts, test targeted contract/loyalty offers.")
print("3. If complaints are associated with churn, improve complaint resolution and follow-up.")
print("4. If usage trends differ by churn, test proactive service checks or relevant plan recommendations.")
print("These are hypotheses for retention experiments, not causal conclusions from observational data.")

## 15. Final objective checklist and export

In [ ]:
objective_checklist = pd.DataFrame({
    "Objective": [
        "Inspect each raw dataset", "Visualize raw distributions", "Clean duplicate customer records",
        "Aggregate usage to customer-level features", "Merge all three datasets", "Quantify class imbalance",
        "Apply SMOTE only to training data", "Compare three algorithms", "Evaluate with multiple metrics",
        "Plot confusion matrices and ROC/PR curves", "Rank influential features", "Analyze churn behaviour and recommendations"
    ],
    "Evidence in notebook": ["Yes", "Yes", "Yes (with documented rule)", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes"]
})
display(objective_checklist)

# Save reproducible outputs to the current working directory.
df.to_csv("telecom_churn_customer_level.csv", index=False)
results.to_csv("model_comparison_metrics.csv", index=False)
if importance is not None:
    importance.rename("importance").to_csv("feature_importance.csv")
print("Saved: telecom_churn_customer_level.csv, model_comparison_metrics.csv" + (", feature_importance.csv" if importance is not None else ""))

## 16. Suggested final conclusion template

After running all cells, write the conclusion using your actual metric table and charts:

- The data contains 10,000 unique customers with 12 monthly usage observations per customer (confirm in output).
- The customer-level churn percentage is shown in the class-imbalance section; do not quote the monthly-row percentage as if it were customer-level.
- SMOTE was applied only to the training data; the test set kept its original class distribution.
- The model was selected by test PR-AUC with recall and F1 as tie-breakers. Use the printed results to state which algorithm performed best; do not assume a model in advance.
- Report the chosen model's precision, recall, F1, ROC-AUC and PR-AUC, and explain the trade-off between catching churners and generating false alarms.
- Use the feature importance and behavioural group charts to state which variables/groups are associated with churn. Avoid causal claims.

### Limitations to mention
1. Customer-info duplicates with conflicting signup dates were resolved using a most-complete-record rule; ambiguous cases should be verified with a source of truth.
2. SMOTE may improve recall but can also reduce precision; model choice should reflect the business cost of false negatives and false positives.
3. If the aim is to predict future churn, construct features only from months before a fixed cutoff and evaluate on a later time period.
4. Feature importance is predictive association, not proof that changing a feature will reduce churn.